# 00a. Project setup and metadata

このノートブックは、RNA-seq解析の「地図」を作る段階である。まだ統計解析はしない。

**この段階が答える問い**

- どのサンプルがあるか。
- どの群を比較したいか。
- 後続ノートブックがどの設定ファイルを見るか。

**入力**

- `raw_data/` のFASTQファイル
- この実験の設計メモ

**出力**

- `metadata/samples.tsv`
- `metadata/contrasts.tsv`
- `config/analysis_config.json`

**次に進む条件**

- サンプル名、群、replicate、FASTQパスが正しい。
- `Non` を基準群にしてよい、という前提を一度確認した。


## RNA-seq解析の全体像

```mermaid
flowchart LR
  A["sample design"] --> B["FASTQ QC（Quality Control）"]
  B --> C["reference setup"]
  C --> D["Salmon quantification"]
  D --> E["gene count matrix"]
  E --> F["sample QC（Quality Control）"]
  F --> G["DESeq2"]
  G --> H["biological interpretation"]
```

初心者が混乱しやすい点は、`FASTQ`、`参照ファイル`、`count matrix`、`DEGリスト` が別物だという点である。

- FASTQ: シーケンサーから来たreadそのもの。
- 参照ファイル: readを遺伝子・転写産物に対応づける辞書。
- count matrix: サンプルごとに各遺伝子が何read相当読まれたかの表。
- DEGリスト: 統計モデルで群間差を検定した結果。


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
RAW_DATA_DIR = PROJECT_DIR / "raw_data"
METADATA_DIR = PROJECT_DIR / "metadata"
CONFIG_DIR = PROJECT_DIR / "config"
RESULTS_DIR = PROJECT_DIR / "results"

for path in [METADATA_DIR, CONFIG_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PROJECT_DIR


## 解析に必要な環境（ライブラリ・ツール）の診断

RNA-seq解析には、Pythonモジュール、統計解析用のRパッケージ、およびmappingやアセンブリを行う外部コマンドラインツール（CLIツール）が必要である。
ここで主要なライブラリやツールが揃っているか確認する。


In [ ]:
# Diagnostic check for required software, Python modules, and R libraries
import shutil
import subprocess
import sys
import pandas as pd

diagnostic_results = []

# 1. Check Python Libraries
py_modules = {
    "pandas": "Data manipulation",
    "numpy": "Numerical computation",
    "matplotlib": "Plotting/visualization",
    "seaborn": "Statistical visualization",
    "sklearn": "Machine learning (scikit-learn)",
}
for mod, desc in py_modules.items():
    try:
        m = __import__(mod)
        ver = getattr(m, "__version__", "Available")
        diagnostic_results.append({"Category": "Python", "Name": mod, "Available": True, "Details": f"version {ver} ({desc})"})
    except ImportError:
        diagnostic_results.append({"Category": "Python", "Name": mod, "Available": False, "Details": f"Missing! ({desc})"})

# 2. Check R Packages
r_packages = {
    "DESeq2": "Differential expression analysis",
    "clusterProfiler": "Pathway enrichment analysis",
    "org.Hs.eg.db": "Human gene annotation database",
    "org.Mm.eg.db": "Mouse gene annotation database",
    "pheatmap": "Heatmap visualization",
    "ggplot2": "Advanced plotting",
}
rscript_path = shutil.which("Rscript")
if rscript_path:
    for pkg, desc in r_packages.items():
        res = subprocess.run([rscript_path, "-e", f"library({pkg})"], capture_output=True, text=True)
        if res.returncode == 0:
            diagnostic_results.append({"Category": "R Package", "Name": pkg, "Available": True, "Details": f"Available ({desc})"})
        else:
            diagnostic_results.append({"Category": "R Package", "Name": pkg, "Available": False, "Details": f"Failed to load! ({desc})"})
else:
    for pkg, desc in r_packages.items():
        diagnostic_results.append({"Category": "R Package", "Name": pkg, "Available": False, "Details": "Rscript command not found in PATH"})

# 3. Check CLI Tools
cli_tools = {
    "fastqc": "Quality control of raw reads",
    "multiqc": "Aggregate QC reports",
    "salmon": "Transcript abundance quantification",
    "STAR": "Genome mapping",
    "samtools": "BAM files processing",
    "featureCounts": "Summarize read counts over genomic features",
    "trinity": "De novo transcriptome assembly (optional)",
}
for tool, desc in cli_tools.items():
    path = shutil.which(tool)
    if path:
        diagnostic_results.append({"Category": "CLI Tool", "Name": tool, "Available": True, "Details": f"{path} ({desc})"})
    else:
        diagnostic_results.append({"Category": "CLI Tool", "Name": tool, "Available": False, "Details": f"Not found in PATH! ({desc})"})

# Display as a formatted table
df_diag = pd.DataFrame(diagnostic_results)
display(df_diag)

# Warning if anything critical is missing
missing_critical = df_diag[~df_diag["Available"] & (df_diag["Name"] != "trinity")]
if not missing_critical.empty:
    print("WARNING: Some critical components are missing from your environment!")
    print("Please make sure you have activated the correct Conda environment (e.g. 'rna-seq').")
else:
    print("SUCCESS: All major environment components are verified and ready!")


## まずFASTQファイルが見えるか確認する

ここでは中身を解析せず、ファイルの場所だけ確認する。`*_1.fq.gz` と `*_2.fq.gz` がペアで並んでいることが大事である。


In [ ]:
fastq_files = sorted(RAW_DATA_DIR.glob("*/*.fq.gz"))
print(f"FASTQ files found: {len(fastq_files)}")
for path in fastq_files[:20]:
    print(path.relative_to(PROJECT_DIR))


## FASTQの中身を少し見る

FASTQは巨大なテキストファイルなので、全部を開くのではなく、先頭の数readだけを見る。

1 readは必ず4行である。

$$
1\ \mathrm{read} = \mathrm{header} + \mathrm{sequence} + \mathrm{separator} + \mathrm{quality}
$$

- 1行目: read名。`@` で始まる。
- 2行目: 塩基配列。`A/T/G/C/N` が並ぶ。
- 3行目: 区切り。通常は `+`。
- 4行目: quality score。各塩基の信頼度を文字で表す。

ここでは `NAC_S2_2h_1` のR1を例に、先頭3 readだけ表示する。


In [ ]:
import gzip

example_fastq = PROJECT_DIR / "raw_data/NAC_S2_2h_1/NAC_S2_2h_1_1.fq.gz"
N_READS_TO_SHOW = 3

with gzip.open(example_fastq, "rt") as handle:
    for read_index in range(1, N_READS_TO_SHOW + 1):
        header = handle.readline().rstrip("\n")
        sequence = handle.readline().rstrip("\n")
        separator = handle.readline().rstrip("\n")
        quality = handle.readline().rstrip("\n")

        print(f"--- read {read_index} ---")
        print("header   :", header)
        print("sequence :", sequence[:80] + ("..." if len(sequence) > 80 else ""))
        print("separator:", separator)
        print("quality  :", quality[:80] + ("..." if len(quality) > 80 else ""))
        print("length   :", len(sequence), "bases")
        print()


## sample table

`samples.tsv` は「どのFASTQがどのサンプルか」を後続ノートブックへ伝える表である。

`condition` はDESeq2で比較する群名である。ここでは現在の資料に合わせて `Non`、`Oxi_2h`、`NAC_S2_2h` を使う。


In [ ]:
sample_rows = [
    {
        "sample_id": "NAC_S2_2h_1",
        "condition": "NAC_S2_2h",
        "treatment": "NAC_S2",
        "timepoint": "2h",
        "replicate": 1,
        "fastq_1": "raw_data/NAC_S2_2h_1/NAC_S2_2h_1_1.fq.gz",
        "fastq_2": "raw_data/NAC_S2_2h_1/NAC_S2_2h_1_2.fq.gz",
    },
    {
        "sample_id": "NAC_S2_2h_2",
        "condition": "NAC_S2_2h",
        "treatment": "NAC_S2",
        "timepoint": "2h",
        "replicate": 2,
        "fastq_1": "raw_data/NAC_S2_2h_2/NAC_S2_2h_2_1.fq.gz",
        "fastq_2": "raw_data/NAC_S2_2h_2/NAC_S2_2h_2_2.fq.gz",
    },
    {
        "sample_id": "NAC_S2_2h_3",
        "condition": "NAC_S2_2h",
        "treatment": "NAC_S2",
        "timepoint": "2h",
        "replicate": 3,
        "fastq_1": "raw_data/NAC_S2_2h_3/NAC_S2_2h_3_1.fq.gz",
        "fastq_2": "raw_data/NAC_S2_2h_3/NAC_S2_2h_3_2.fq.gz",
    },
    {
        "sample_id": "Non_1",
        "condition": "Non",
        "treatment": "Non",
        "timepoint": "baseline",
        "replicate": 1,
        "fastq_1": "raw_data/Non_1/Non_1_1.fq.gz",
        "fastq_2": "raw_data/Non_1/Non_1_2.fq.gz",
    },
    {
        "sample_id": "Non_2",
        "condition": "Non",
        "treatment": "Non",
        "timepoint": "baseline",
        "replicate": 2,
        "fastq_1": "raw_data/Non_2/Non_2_1.fq.gz",
        "fastq_2": "raw_data/Non_2/Non_2_2.fq.gz",
    },
    {
        "sample_id": "Non_3",
        "condition": "Non",
        "treatment": "Non",
        "timepoint": "baseline",
        "replicate": 3,
        "fastq_1": "raw_data/Non_3/Non_3_1.fq.gz",
        "fastq_2": "raw_data/Non_3/Non_3_2.fq.gz",
    },
    {
        "sample_id": "Oxi_2h_1",
        "condition": "Oxi_2h",
        "treatment": "Oxi",
        "timepoint": "2h",
        "replicate": 1,
        "fastq_1": "raw_data/Oxi_2h_1/Oxi_2h_1_1.fq.gz",
        "fastq_2": "raw_data/Oxi_2h_1/Oxi_2h_1_2.fq.gz",
    },
    {
        "sample_id": "Oxi_2h_2",
        "condition": "Oxi_2h",
        "treatment": "Oxi",
        "timepoint": "2h",
        "replicate": 2,
        "fastq_1": "raw_data/Oxi_2h_2/Oxi_2h_2_1.fq.gz",
        "fastq_2": "raw_data/Oxi_2h_2/Oxi_2h_2_2.fq.gz",
    },
    {
        "sample_id": "Oxi_2h_3",
        "condition": "Oxi_2h",
        "treatment": "Oxi",
        "timepoint": "2h",
        "replicate": 3,
        "fastq_1": "raw_data/Oxi_2h_3/Oxi_2h_3_1.fq.gz",
        "fastq_2": "raw_data/Oxi_2h_3/Oxi_2h_3_2.fq.gz",
    },
]

samples_path = METADATA_DIR / "samples.tsv"
header = list(sample_rows[0])
with samples_path.open("w", encoding="utf-8") as out:
    out.write("\t".join(header) + "\n")
    for row in sample_rows:
        out.write("\t".join(str(row[column]) for column in header) + "\n")

print(samples_path.relative_to(PROJECT_DIR))


In [ ]:
import pandas as pd

samples = pd.read_csv(METADATA_DIR / "samples.tsv", sep="\t")
samples


## FASTQパスの検査

この検査は、表に書いたFASTQパスが実際に存在するかを見るだけである。ここで失敗する場合、後続のQCやSalmonは必ず失敗する。


In [ ]:
missing = []
for _, row in samples.iterrows():
    for column in ["fastq_1", "fastq_2"]:
        path = PROJECT_DIR / row[column]
        if not path.exists():
            missing.append(str(path.relative_to(PROJECT_DIR)))

if missing:
    print("Missing FASTQ paths:")
    for path in missing:
        print(" -", path)
else:
    print("All FASTQ paths in metadata exist.")


## contrast table

`contrasts.tsv` は「どの群とどの群を比べるか」を書く表である。

`test_condition` が分子、`reference_condition` が基準である。たとえば `Oxi_2h_vs_Non` は、`Oxi_2h` が `Non` より上がるか下がるかを見る。


In [ ]:
contrast_rows = [
    {
        "contrast_id": "Oxi_2h_vs_Non",
        "test_condition": "Oxi_2h",
        "reference_condition": "Non",
        "description": "Oxidative stress at 2h compared with untreated control",
    },
    {
        "contrast_id": "NAC_S2_2h_vs_Non",
        "test_condition": "NAC_S2_2h",
        "reference_condition": "Non",
        "description": "NAC_S2 at 2h compared with untreated control",
    },
    {
        "contrast_id": "NAC_S2_2h_vs_Oxi_2h",
        "test_condition": "NAC_S2_2h",
        "reference_condition": "Oxi_2h",
        "description": "NAC_S2 at 2h compared with oxidative stress at 2h",
    },
]

contrasts_path = METADATA_DIR / "contrasts.tsv"
header = list(contrast_rows[0])
with contrasts_path.open("w", encoding="utf-8") as out:
    out.write("\t".join(header) + "\n")
    for row in contrast_rows:
        out.write("\t".join(str(row[column]) for column in header) + "\n")

contrasts = pd.read_csv(contrasts_path, sep="\t")
contrasts


## central config

`analysis_config.json` は、ノートブック間で共有する設定メモである。

重要なのは、参照ファイルの欄である。ここに書くパスは「これから作る予定の参照辞書」を指す。実ファイルは次の `00b_reference_setup_gencode_grch38.ipynb` で作る。


In [ ]:
analysis_config = {
    "project_dir": str(PROJECT_DIR),  # Setting to the absolute project root used by every notebook.
    "raw_data_dir": "raw_data",  # Setting to the directory containing FASTQ sample folders.
    "samples_path": "metadata/samples.tsv",  # Setting to the sample metadata table.
    "contrasts_path": "metadata/contrasts.tsv",  # Setting to the DESeq2 group comparison table.
    "organism": "human",  # Setting to the organism assumption used for reference and enrichment.
    "gene_id_type": "SYMBOL",  # Setting to the gene ID type used by clusterProfiler.
    "reference": {
        "gencode_release": "49",  # Setting to the GENCODE release used for all reference files.
        "transcript_fasta": "reference/gencode_grch38/gencode.v49.transcripts.fa.gz",  # Setting to the transcript FASTA used to build the Salmon index.
        "salmon_index": "reference/gencode_grch38/salmon_index",  # Setting to the Salmon transcriptome index directory.
        "tx2gene_path": "reference/gencode_grch38/tx2gene.tsv",  # Setting to the transcript-to-gene mapping table.
        "gtf_path": "reference/gencode_grch38/gencode.v49.annotation.gtf.gz",  # Setting to the annotation GTF matching the transcript FASTA.
        "genome_fasta": "",  # Setting to the genome FASTA; not required for this Salmon-first workflow.
    },
    "quantification": {
        "method": "salmon",  # Setting to the quantification backend used in notebook 02.
        "threads": 8,  # Setting to the maximum CPU threads for external tools.
        "salmon": {
            "library_type": "A",  # Setting to Salmon automatic library type detection.
            "validate_mappings": True,  # Setting to fail early if sample IDs and file paths do not match.
        },
        "star": {
            "read_length": 150,  # Setting to the read length in base pairs.
            "sjdb_overhang": 149,  # Setting to sjdbOverhang (read_length - 1).
            "featurecounts_strandness": 0,  # Setting to featureCounts strand specificity (0: unstranded, 1: stranded, 2: reversely stranded).
        },
        "trinity": {
            "max_memory": "10G",  # Setting to the maximum memory allocated for Trinity.
            "seq_type": "fq",  # Setting to the sequence format (e.g. fq for fastq).
            "ss_lib_type": "RF",  # Setting to the strand-specific library type (e.g. RF, FR, or empty).
        },
    },
    "differential_expression": {
        "count_matrix_path": "results/counts/gene_counts.tsv",  # Setting to the gene-level count matrix consumed by DESeq2.
        "design_formula": "~ condition",  # Setting to the DESeq2 design formula.
        "reference_condition": "Non",  # Setting to the baseline condition for factor releveling.
        "min_count": 10,  # Setting to the minimum raw count used before DESeq2.
        "min_samples": 2,  # Setting to the number of samples that must pass min_count.
        "alpha": 0.05,  # Setting to the FDR threshold for significant genes.
        "lfc_threshold": 1.0,  # Setting to the absolute log2 fold-change threshold used in plots/reports.
    },
}

config_path = CONFIG_DIR / "analysis_config.json"
config_path.write_text(json.dumps(analysis_config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(config_path.relative_to(PROJECT_DIR))


In [ ]:
config = json.loads((CONFIG_DIR / "analysis_config.json").read_text(encoding="utf-8"))
config


## この段階の確認

ここまでで「解析対象のリスト」と「比較したい群」が決まりした。次は、FASTQ readを転写産物・遺伝子に対応づけるための参照辞書を作る。

**よくある間違い**

- `sample_id` とFASTQフォルダ名がずれている。
- `condition` の表記揺れがある。例: `Oxi_2h` と `Oxi-2h`。
- 基準群が意図と違う。ここでは `Non` を基準にしている。

**小さい練習**

`samples["condition"].value_counts()` を実行して、各群が3 replicateあることを確認しよう。


In [ ]:
samples["condition"].value_counts()
